In [ ]:
!nvidia-smi

Thu Mar 26 02:41:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers datasets peft bitsandbytes pymatgen pandas numpy matplotlib tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.1 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609

In [ ]:
import torch, gc

# Delete old model and free CUDA memory
#del model
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

CUDA available: True
Device: cuda


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

CSV_PATH = "/content/zif8_synthesis_data.csv"
df = pd.read_csv(CSV_PATH)
print(df.head())



  Solvent_1 Solvent_2  Ratio  Temperature  Duration Quality
0      MeOH       H2O    2.0           60       3.0    High
1      MeOH       H2O    0.5           60       3.0    High
2      MeOH      MeOH    1.0          100      12.0    High
3      MeOH       H2O    3.0           60       3.0    High
4      MeOH       H2O    4.0           60       3.0    High


In [ ]:
X = df[["Solvent_1",
        "Solvent_2",
        "Ratio",
        "Temperature",
        "Duration"]]

y = df["Quality"]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (68, 5)
y shape: (68,)


In [ ]:
import os
import json
import gc
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model


# =========================================================
# 0. PREPARE PROMPT DATA
# =========================================================
def build_prompt(row):
    prompt = (
        "Instruction: Predict synthesis quality from synthesis conditions.\n\n"
        "Input:\n"
        f"Solvent_1 = {row['Solvent_1']}\n"
        f"Solvent_2 = {row['Solvent_2']}\n"
        f"Ratio = {row['Ratio']}\n"
        f"Temperature = {row['Temperature']}\n"
        f"Duration = {row['Duration']}\n\n"
        "Output:"
    )
    return pd.Series({
        "prompt": prompt,
        "response": str(row["Quality"])
    })

prompt_df = df.apply(build_prompt, axis=1)
prompt_df_llm = prompt_df.rename(columns={"response": "target"})


# =========================================================
# 1. PATH + CONFIG
# =========================================================
PROJECT_ROOT = Path(".")
PROCESSED_DIR = PROJECT_ROOT / "processed_zif8"
PROCESSED_DIR.mkdir(exist_ok=True)

model_name = "microsoft/phi-2"
MAX_LEN = 512

from sklearn.model_selection import StratifiedKFold
cv = 10
RANDOM_STATE = 1
skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)


def save_jsonl(samples, path):
    with path.open("w", encoding="utf-8") as f:
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")


# =========================================================
# 2. TOKENIZER
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# =========================================================
# 3. TOKENIZE FUNCTION
# =========================================================
def tokenize_example(example):
    prompt = example["prompt"]
    response = example["target"]

    full_text = f"### Instruction:\n{prompt}\n\n### Response:\n{response}"

    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

    prompt_text = f"### Instruction:\n{prompt}\n\n### Response:\n"
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

    labels = tokenized["input_ids"].copy()
    prompt_len = len(prompt_tokens["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len
    tokenized["labels"] = labels

    return tokenized


# =========================================================
# 4. DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorForCausalLM:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        labels = [f["labels"] for f in features]

        features_no_labels = []
        for f in features:
            f_copy = {k: v for k, v in f.items() if k != "labels"}
            features_no_labels.append(f_copy)

        batch = DataCollatorWithPadding(
            tokenizer=self.tokenizer,
            padding=True,
            return_tensors="pt"
        )(features_no_labels)

        max_len = batch["input_ids"].shape[1]

        padded_labels = []
        for l in labels:
            l = l[:max_len]
            pad_len = max_len - len(l)
            padded_labels.append(l + [-100] * pad_len)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch


data_collator = DataCollatorForCausalLM(tokenizer)


# =========================================================
# 5. PREDICT FUNCTION
# =========================================================
def predict_quality(model, tokenizer, solvent_1, solvent_2, ratio, temperature, duration):
    prompt_str = (
        "Instruction: Predict synthesis quality from synthesis conditions.\n\n"
        "Input:\n"
        f"Solvent_1 = {solvent_1}\n"
        f"Solvent_2 = {solvent_2}\n"
        f"Ratio = {ratio}\n"
        f"Temperature = {temperature}\n"
        f"Duration = {duration}\n\n"
        "Output:"
    )

    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt_str,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    predicted_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    predicted_response = predicted_response.split()[0] if predicted_response else ""

    if predicted_response in ["High", "Low"]:
        return predicted_response
    return None


# =========================================================
# 6. RUN 10 TIMES
# =========================================================
all_results = []
all_conf_mats = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print("=" * 80)
    print(f"RUN fold = {fold}/{cv}")

    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test  = y.iloc[test_idx].copy()

    print(f"Train size = {len(train_idx)}")
    print(f"Test size  = {len(test_idx)}")

    train_samples = prompt_df_llm.iloc[train_idx].to_dict(orient='records')
    test_samples  = prompt_df_llm.iloc[test_idx].to_dict(orient='records')

    train_path = PROCESSED_DIR / f"train_fold{fold}.jsonl"
    test_path  = PROCESSED_DIR / f"test_fold{fold}.jsonl"

    save_jsonl(train_samples, train_path)
    save_jsonl(test_samples, test_path)


    # -----------------------------------------------------
    # Load dataset
    # -----------------------------------------------------
    raw_datasets = load_dataset(
        "json",
        data_files={
            "train": str(train_path),
            "test": str(test_path),
        }
    )

    tokenized_datasets = raw_datasets.map(
        tokenize_example,
        remove_columns=raw_datasets["train"].column_names
    )

    # -----------------------------------------------------
    # Load model
    # -----------------------------------------------------
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    config = AutoConfig.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    if not hasattr(config, "pad_token_id") or config.pad_token_id is None:
        config.pad_token_id = tokenizer.pad_token_id

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        config=config,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    base_model.config.use_cache = False

    # -----------------------------------------------------
    # LoRA
    # -----------------------------------------------------
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "dense",
            "fc1",
            "fc2",
        ],
    )

    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    # -----------------------------------------------------
    # Training arguments
    # -----------------------------------------------------
    output_dir = str(PROJECT_ROOT / f"phi2_quality_lora_rs{fold}")

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=30,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        data_collator=data_collator,
    )

    trainer.train()

    # -----------------------------------------------------
    # Evaluate on test set
    # -----------------------------------------------------
    test_df = X_test.copy()
    test_df["Quality"] = y_test

    y_true_eval = []
    y_pred_eval = []

    for _, r in test_df.iterrows():
        pred = predict_quality(
            trainer.model,
            tokenizer,
            r["Solvent_1"],
            r["Solvent_2"],
            r["Ratio"],
            r["Temperature"],
            r["Duration"]
        )

        if pred is None:
            continue

        y_true_eval.append(r["Quality"])
        y_pred_eval.append(pred)

    if len(y_true_eval) == 0:
        print("Không có prediction hợp lệ ở vòng này.")
        continue

    acc = accuracy_score(y_true_eval, y_pred_eval)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true_eval,
        y_pred_eval,
        average="weighted",
        zero_division=0
    )
    cm = confusion_matrix(y_true_eval, y_pred_eval, labels=["High", "Low"])

    print(f"\nAccuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true_eval, y_pred_eval, zero_division=0))

    print("\nConfusion Matrix [High, Low]:")
    print(cm)

    all_results.append({
        "random_state": fold,
        "accuracy": acc,
        "precision_weighted": prec,
        "recall_weighted": rec,
        "f1_weighted": f1,
        "n_eval_samples": len(y_true_eval),
    })
    all_conf_mats.append(cm)

    # -----------------------------------------------------
    # Free memory
    # -----------------------------------------------------
    del trainer
    del model
    del base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =========================================================
# 7. SUMMARY
# =========================================================
print("\n" + "=" * 90)
print("RESULTS OF ALL RUNS")

results_df = pd.DataFrame(all_results)
print(results_df)

if len(results_df) > 0:
    print("\n" + "=" * 90)
    print("AVERAGE OVER 10 RUNS")
    print(f"Average Accuracy  : {results_df['accuracy'].mean():.4f}")
    print(f"Average Precision : {results_df['precision_weighted'].mean():.4f}")
    print(f"Average Recall    : {results_df['recall_weighted'].mean():.4f}")
    print(f"Average F1-score  : {results_df['f1_weighted'].mean():.4f}")

    avg_cm = np.mean(np.array(all_conf_mats), axis=0)
    sum_cm = np.sum(np.array(all_conf_mats), axis=0)

    print("\nAverage Confusion Matrix [High, Low]:")
    print(avg_cm)

    print("\nSummed Confusion Matrix [High, Low]:")
    print(sum_cm)
else:
    print("Không có run nào tạo được prediction hợp lệ.")

RUN fold = 1/10
Train size = 61
Test size  = 7


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/61 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

trainable params: 23,592,960 || all params: 2,803,276,800 || trainable%: 0.8416


Step,Training Loss
10,3.937631
20,0.523397
30,0.448969
40,0.339944
50,0.181985
60,0.151507
70,0.075256
80,0.131855
90,0.079608
100,0.073382



Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1-score  : 1.0000

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00         5
         Low       1.00      1.00      1.00         1

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6


Confusion Matrix [High, Low]:
[[5 0]
 [0 1]]

RESULTS OF ALL RUNS
   random_state  accuracy  precision_weighted  recall_weighted  f1_weighted  \
0             1  0.857143            0.928571         0.857143     0.874459   
1             2  0.857143            0.928571         0.857143     0.874459   
2             3  0.714286            0.510204         0.714286     0.595238   
3             4  1.000000            1.000000         1.000000     1.000000   
4             5  1.000000            1.000000         1.000000     1.000000   
5             6  0.428571      